In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

True
NVIDIA GeForce RTX 5050 Laptop GPU
13.0


In [ ]:
import pandas as pd
df = pd.read_csv("b_hashed_list.csv")

In [ ]:
df["Date of document:"] = (
    df["Date of document:"]
    .astype(str)
    .str.replace(r"[:;].*$", "", regex=True)
    .str.strip()
)
df.columns = df.columns.str.replace(":", "")
df.to_csv("b_hashed_list.csv")

In [ ]:
import pandas as pd

df = pd.read_csv(
    "b_hashed_list.csv",
    usecols=[
        "title",
        "ref",
        "status",
        "CELEX number",
        "Author",
        "Date of document",
        "link",
        "Latest consolidated version",
        "hash_id"
    ]
)
#clean metadata
df = df.set_index("hash_id")

In [ ]:
import pandas as pd
from pathlib import Path

from llama_index.core import SimpleDirectoryReader, 
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from dotenv import load_dotenv
import os

load_dotenv()

qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")


# 1. Charger le DataFrame filtré
df = pd.read_csv(
    "b_hashed_list.csv",
    usecols=[
        "title",
        # "ref",
        "status",
        "CELEX number:",
        "Author:",
        "Date of document:",
        "link",
        # "Latest consolidated version",
        "hash_id"
    ]
)
df = df.set_index("hash_id")

In [27]:
print(qdrant_url)

https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/


In [ ]:
# 2. Ajout métadonnées → Document
def add_file_metadata(path):
    p = Path(path)
    hash_id = p.stem
    row = df.loc[hash_id].to_dict()
    return {"hash_id": hash_id, **row}

embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

# 3. Charger documents
documents = SimpleDirectoryReader(
    "texts/",
    file_metadata=add_file_metadata
).load_data()

splitter = SemanticSplitterNodeParser(
    buffer_size=1,            # Nombre de phrases avant et après la rupture à inclure
    breakpoint_percentile_window=5, # %ile pour définir un seuil de 'rupture sémantique'
    embed_model=embed_model
)
# splitter = SentenceSplitter(chunk_size=2048, chunk_overlap=50)

nodes = splitter.get_nodes_from_documents(documents)


# 5. Qdrant setup
client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="eurlex"
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)

2025-11-16 17:20:12,854 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2025-11-16 17:20:16,225 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/ "HTTP/1.1 200 OK"
2025-11-16 17:20:16,432 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/eurlex/exists "HTTP/1.1 200 OK"
2025-11-16 17:20:16,518 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/eurlex "HTTP/1.1 200 OK"
2025-11-16 17:20:31,928 - INFO - HTTP Request: PUT https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/eurlex/points?wait=true "HTTP/1.1 200 OK"
2025-11-16 17:20:32,152 - INFO - HTTP Request: PUT https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/eurlex/points?wait=true "HTTP/1.1 200 OK"
202

In [ ]:
import os
import pandas as pd
from pathlib import Path

from llama_index.core import SimpleDirectoryReader, StorageContext, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.mistralai import MistralEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient


# =========================
# CONFIGURATION
# =========================

# Variables d'environnement requises :
# export MISTRAL_API_KEY="xxx"
# export QDRANT_URL="http://localhost:6333"
# export QDRANT_API_KEY="xxx"

MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY")
qdrant_url = os.environ.get("QDRANT_URL")
qdrant_api_key = os.environ.get("QDRANT_API_KEY")

# Charger ton dataframe contenant les métadonnées
df = pd.read_csv("metadata.csv", index_col=0)


# =========================
# 1. Ajout métadonnées → Document
# =========================

def add_file_metadata(path: str):
    p = Path(path)
    hash_id = p.stem
    row = df.loc[hash_id].to_dict()
    return {"hash_id": hash_id, **row}


# =========================
# 2. Charger les documents
# =========================

documents = SimpleDirectoryReader(
    "texts/",
    file_metadata=add_file_metadata
).load_data()


# =========================
# 3. Découpage en chunks
# =========================

splitter = SentenceSplitter(
    chunk_size=2048,
    chunk_overlap=50
)

nodes = splitter.get_nodes_from_documents(documents)


# =========================
# 4. Modèle d'embedding Mistral
# =========================

embed_model = MistralEmbedding(
    model="mistral-embed"
)

Settings.embed_model = embed_model


# =========================
# 5. Qdrant setup
# =========================

client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="eurlex"
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)


# =========================
# 6. Indexation
# =========================

from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(
    nodes,
    storage_context=storage_context
)

print("✅ Indexation terminée avec Mistral embeddings")
